# Next step: step 8c, the output index
**What it adds:** `<output>/index.json`, the output folder's table of contents.

```
<output>/
    index.json          "models": id -> 14 parameters, every model ever saved here
                        "runs":   one record per run that saved here
    model_000000/       one folder per model id (was: per row of the run)
        model.json
        simulations/
    model_000001/
```

- **Models are matched, not numbered by row.** Each row of a run's `models` is matched exactly (all 14 values) against
  the index. A model already there uses its folder; a new one gets the next id at the bottom. So a reordered list, or a
  GA generation that shares some individuals with the last one, lands in the same folders and reuses what's saved.
- **`result.ids`:** row m's model id (its folder is `model_<id>`). `None` for a row the folder doesn't have when the run
  only reads, and every row without an output folder.
- **Each run that saves is recorded:** when it started and finished, the code, what it returned and saved, the full
  simulator configs (image and fitness configs later), and `model_ids` (row m's id, so later per-row statuses always
  point at the right model). The record is written when the run starts and again when it ends, so a run that crashed
  shows `"finished": null`. Step 9 adds each simulation's status, time and error.
- **A run that only reads writes nothing.** One run at a time per output folder.

## What changed
**`src/galactic_tidal_constraints/simr/simr.py`.** Before the loop, rows are matched against the index and the run's
record is written; after it, the record is completed:
```python
index = _open_index(Path(output)) if output is not None else None
ids = _model_ids(index, models, add=bool(saved)) if index is not None else [None] * len(models)
if saved:
    run_record = {"started": _now(), "finished": None, "code": versions,
                  "run": {"return": returned, "save": saved},
                  "simulator": [_json_ready(asdict(config)) for config in configs],
                  "image": [], "fitness": [], "model_ids": ids}
    index["runs"].append(run_record)
    _write_index(Path(output), index)                 # atomic, like every other file
for m, params in enumerate(models):
    folder = Path(output) / f"model_{ids[m]}" if ids[m] is not None else None
    ...
if saved:
    run_record["finished"] = _now()
    _write_index(Path(output), index)
```
Matching: the 14 values as a key; JSON writes floats so they read back exactly:
```python
def _model_ids(index, models, add):
    known = {tuple(params.values()): model_id for model_id, params in index["models"].items()}
    next_id = max((int(model_id) for model_id in index["models"]), default=-1) + 1
    for params in models:
        key = tuple(model_parameters(params).values())
        if key not in known and add:
            known[key] = f"{next_id:06d}"      # a new model: the next id at the bottom
            ...
```
`simr.load_index(output)` reads it back. **Tests:** `tests/test_simr.py`, section "Step 8c".
**Decision:** `PLAN.md`, Structure (Output index).

In [ ]:
%matplotlib inline
import json
import tempfile
import time
from pathlib import Path

import numpy as np

from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

# Line 3 of JSPAM's archive/data.txt, the same with mass2 doubled, and with vx changed
params = np.array([1.08119, -1.66798, -2.06826, -0.18438, -0.39693, -0.02919, 0.51654, 0.36192, 0.94043, 0.43877,
                   12.97415, 135.11902, 97.54035, 305.01225])
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2
faster = params.copy()
faster[sim.PARAM_INDEX["vx"]] += 0.1
N = 1500
simulators = [{"n1": N, "n2": N, "lnl": 0.0025}, {"n1": N, "n2": N, "lnl": 0.001}]

output = Path(tempfile.mkdtemp(prefix="simr_8c_"))
print("output:", output)

## 1. The first run
Generation 1: two models, saved:

In [ ]:
first = simr.run(np.stack([params, heavier]), {"run": {"save": ["sim.particles"]}, "simulator": simulators}, output=output)
print("ids:", first.ids)
print(sorted(path.name for path in output.iterdir()))

index = simr.load_index(output)
print("\nmodels in the index:", {model_id: model["mass2"] for model_id, model in index["models"].items()}, "(mass2 of each)")
record = index["runs"][0]
print("run 0:", {key: record[key] for key in ("started", "finished", "run", "model_ids", "image", "fitness")})
print("  simulator[0]:", record["simulator"][0])
print("  code:", record["code"])

## 2. A second run, reordered, with a new model
Generation 2 shares both models with generation 1, in the other order, and adds one. The known models keep their
folders, and their particles are loaded instead of simulated; only the new model runs. It gets id `000002`, at the
bottom:

In [ ]:
start = time.perf_counter()
second = simr.run(np.stack([heavier, faster, params]),
                  {"run": {"return": ["sim.particles"], "save": ["sim.particles"]}, "simulator": simulators},
                  output=output)
print(f"{time.perf_counter() - start:.1f} s (one new model x 2 entries simulated; the rest loaded)")
print("ids:", second.ids)

index = simr.load_index(output)
print("models in the index:", list(index["models"]))
for r, run_record in enumerate(index["runs"]):
    print(f"run {r}: model_ids {run_record['model_ids']}, started {run_record['started']}")
print("row 2 is model_000000's particles:",
      np.array_equal(second.data["sim.particles"][2, 0], simr.load_simulation(output / "model_000000", "sim_00")["sim.particles"]))

## 3. A run that only reads
No `save`: models the folder has are loaded, the others are simulated, and nothing is written, not even a run record:

In [ ]:
before = (output / "index.json").read_text()
other = params.copy()
other[sim.PARAM_INDEX["vy"]] += 0.1
read = simr.run(np.stack([params, other]), {"run": {"return": ["sim.orbit"]}, "simulator": simulators[:1]}, output=output)
print("ids:", read.ids, " (None: the folder doesn't have that model)")
print("index unchanged:", (output / "index.json").read_text() == before)

## 4. The whole index

In [ ]:
text = (output / "index.json").read_text()
print(f"{len(text) / 1e3:.1f} kB")
print(text[:1500], "...")